# Streaming LLM Client

An AI coding agent needs to talk to a language model — but not with a simple request-response cycle. We need **streaming**: the model sends tokens as they are generated, so the user sees output in real time rather than staring at a blank screen for 30 seconds. We also need to handle **tool calls**, where the model asks to invoke functions like reading files or running shell commands.

In this notebook, we build the foundational layer: a streaming LLM client that talks to [OpenRouter](https://openrouter.ai) (an API gateway that provides access to models from Anthropic, OpenAI, Google, and others through a single OpenAI-compatible endpoint). The client produces a stream of typed events — text deltas, tool-call deltas, usage stats — that downstream consumers (the agentic loop, the UI) can process however they want.

We follow the architecture of the [ai-coding-agent](https://github.com/RivaanRanawat/ai-coding-agent) reference implementation, adapted for our project structure.

## Configuration

Everything starts with configuration. We use [Pydantic](https://docs.pydantic.dev/) models for validation and sensible defaults. The key design decision: **secrets come from environment variables, never from code**. The `Config` class reads `OPENROUTER_API_KEY` and `OPENROUTER_BASE_URL` from the environment so that the same code works locally and in CI without any hardcoded credentials.

**ModelConfig.** Specifies which model to call and with what parameters. The `context_window` field is used later for context management (NB04) — when the conversation approaches 80% of this limit, we trigger compaction.

In [ ]:
from notebooks.agent.config import ModelConfig

model_config = ModelConfig()
print(f"Model:          {model_config.name}")
print(f"Temperature:    {model_config.temperature}")
print(f"Context window: {model_config.context_window:,} tokens")

**Config.** The top-level configuration wires together model settings, working directory, approval policy, and environment-based API credentials. The `validate_config()` method catches missing keys and bad paths before we make any API calls.

In [ ]:
from notebooks.agent.config import Config, ApprovalPolicy

config = Config()
print(f"Model:    {config.model_name}")
print(f"Base URL: {config.base_url}")
print(f"API key:  {'***' + config.api_key[-4:] if config.api_key else 'NOT SET'}")
print(f"CWD:      {config.cwd}")
print(f"Approval: {config.approval.value}")

The config is a Pydantic model, so we can override any field at construction time — this makes it easy to experiment with different models or settings in a notebook:

In [ ]:
custom = Config(
    model=ModelConfig(name="google/gemini-2.5-flash-preview", temperature=0.5),
    approval=ApprovalPolicy.AUTO,
)
print(f"Custom model: {custom.model_name}, temp={custom.temperature}")
print(f"Approval:     {custom.approval.value}")

Before making any API calls, we run `validate_config()` to confirm the key is set and the working directory exists:

In [ ]:
errors = config.validate_config()
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print("Config valid!")

:::{.callout-note}
OpenRouter requires an API key, which you get from [openrouter.ai/keys](https://openrouter.ai/keys). Set it as `export OPENROUTER_API_KEY=sk-or-v1-...` in your shell before running the notebook.

:::

## Stream Events

When we stream a response from the LLM, the data arrives in chunks. Rather than processing raw API dictionaries throughout the codebase, we define a clean set of **event types** that the client produces and the agent loop consumes. This is the contract between layers.

**StreamEventType.** The six event types cover the full lifecycle of a single LLM response:

In [ ]:
from notebooks.agent.events import StreamEventType

for evt in StreamEventType:
    print(f"  {evt.value}")

- `TEXT_DELTA` — a chunk of text from the model's response (what the user sees streaming in).
- `TOOL_CALL_START` — the model decided to call a tool; we know the tool name.
- `TOOL_CALL_DELTA` — an incremental chunk of JSON arguments for that tool call.
- `TOOL_CALL_COMPLETE` — the full tool call (name + parsed arguments) is ready to execute.
- `MESSAGE_COMPLETE` — the model finished its response; token usage stats are available.
- `ERROR` — something went wrong (rate limit, connection error, etc.).

**Data carriers.** Each event type has an associated dataclass. The key ones:

In [ ]:
from notebooks.agent.events import TextDelta, TokenUsage, ToolCall, StreamEvent

# TextDelta: a chunk of streamed text
delta = TextDelta(content="Hello, world!")
print(f"TextDelta: '{delta}'")

# TokenUsage: tracks prompt/completion/cached tokens
usage = TokenUsage(prompt_tokens=100, completion_tokens=50, total_tokens=150)
usage2 = TokenUsage(prompt_tokens=200, completion_tokens=100, total_tokens=300, cached_tokens=50)
combined = usage + usage2
print(f"Combined usage: {combined.total_tokens} total, {combined.cached_tokens} cached")

# ToolCall: a completed tool invocation request
tc = ToolCall(call_id="call_abc123", name="read_file", arguments={"path": "main.py"})
print(f"ToolCall: {tc.name}({tc.arguments})")

**StreamEvent.** The envelope that wraps all of these. A single event has a `type` field and then the relevant data field is populated:

In [ ]:
event = StreamEvent(
    type=StreamEventType.TEXT_DELTA,
    text_delta=TextDelta(content="Hello"),
)
print(f"Event type: {event.type.value}")
print(f"Content:    {event.text_delta.content}")

## The LLM Client

The `LLMClient` wraps the `AsyncOpenAI` client and exposes a single method: `chat_completion()`. It is an **async generator** that yields `StreamEvent` objects. The caller does not need to know anything about OpenAI's chunked response format — it just iterates over typed events.

Key implementation details:

1. **Index-based tool-call accumulation.** When the model wants to call tools, it sends the arguments in small JSON fragments across multiple chunks. Each chunk carries an `index` that identifies which tool call it belongs to. We accumulate these fragments in a dict keyed by index, then emit a `TOOL_CALL_COMPLETE` event with the fully parsed arguments once the stream ends.

2. **Retry with exponential back-off.** Rate limits and connection errors get up to 3 retries with $2^n$ second delays. API errors (like invalid model name) fail immediately.

3. **Non-streaming fallback.** Some use cases (like context compaction in NB04) need a simple request-response call. The same method supports `stream=False`.

In [ ]:
from notebooks.agent.client import LLMClient
from notebooks.agent.events import StreamEventType

client = LLMClient(config)
print(f"Client ready: model={client.config.model_name}")
print(f"Base URL:     {client.config.base_url}")

### Streaming text

We send a simple message and stream the response token by token. Notice the `async for` pattern — this is the same pattern the agent loop will use.

In [ ]:
messages = [{"role": "user", "content": "What is 2 + 2? Answer in one sentence."}]

full_text = ""
usage = None

async for event in client.chat_completion(messages):
    if event.type == StreamEventType.TEXT_DELTA:
        print(event.text_delta.content, end="", flush=True)  # <1>
        full_text += event.text_delta.content
    elif event.type == StreamEventType.MESSAGE_COMPLETE:
        usage = event.usage  # <2>
    elif event.type == StreamEventType.ERROR:
        print(f"\nERROR: {event.error}")

print()  # newline after streaming

1. Each `TEXT_DELTA` carries a small chunk (often a single token). We print immediately for real-time output.
2. `MESSAGE_COMPLETE` arrives once at the end with token usage statistics.

In [ ]:
if usage:
    print(f"Prompt tokens:     {usage.prompt_tokens}")
    print(f"Completion tokens: {usage.completion_tokens}")
    print(f"Total tokens:      {usage.total_tokens}")
    print(f"Cached tokens:     {usage.cached_tokens}")

### Streaming with tool calls

The real power of the client is handling **function calling**. We provide tool schemas in the OpenAI format, and the model can decide to call one or more tools instead of (or in addition to) producing text. The client accumulates the streamed JSON argument fragments and emits a `TOOL_CALL_COMPLETE` event with the parsed arguments.

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit",
                },
            },
            "required": ["city"],
        },
    }
]

messages = [
    {"role": "user", "content": "What's the weather in Tokyo?"}
]

In [ ]:
text_chunks = []
completed_tool_calls = []

async for event in client.chat_completion(messages, tools=tools):
    if event.type == StreamEventType.TEXT_DELTA:
        text_chunks.append(event.text_delta.content)
        print(event.text_delta.content, end="", flush=True)

    elif event.type == StreamEventType.TOOL_CALL_START:
        print(f"\n[Tool call starting: {event.tool_call_delta.name}]")

    elif event.type == StreamEventType.TOOL_CALL_COMPLETE:
        tc = event.tool_call
        completed_tool_calls.append(tc)
        print(f"[Tool call complete: {tc.name}({tc.arguments})]")

    elif event.type == StreamEventType.MESSAGE_COMPLETE:
        print(f"\n[Done — finish_reason={event.finish_reason}]")

The model chose to call `get_weather` with the arguments it inferred from the user's message. In the full agent (NB03), we would now execute this tool and feed the result back to the model for another round.

### Non-streaming mode

For internal operations like context compaction (NB04), we need a simple request-response call. The same `chat_completion` method supports `stream=False` — it returns a single `MESSAGE_COMPLETE` event:

In [ ]:
messages = [{"role": "user", "content": "Say hello in exactly 3 words."}]

async for event in client.chat_completion(messages, stream=False):
    if event.type == StreamEventType.MESSAGE_COMPLETE:
        print(f"Response: {event.text_delta.content}")
        print(f"Tokens:   {event.usage.total_tokens if event.usage else 'N/A'}")

### Cleanup

In [ ]:
await client.close()
print("Client closed.")

## Summary

We built three modules that form the foundation of the coding agent:

| Module | What it provides |
|--------|------------------|
| `config.py` | `Config`, `ModelConfig`, `ApprovalPolicy` — Pydantic models with env-based secrets |
| `events.py` | `StreamEvent`, `StreamEventType`, `TextDelta`, `ToolCall`, `TokenUsage` — typed event protocol |
| `client.py` | `LLMClient` — async streaming client with retry, tool-call accumulation, and non-streaming fallback |

The `LLMClient.chat_completion()` async generator is the key abstraction. Any consumer — whether it's an agent loop, a notebook cell, or a Flet UI — processes the same stream of typed events. In the [next notebook](/notebooks/apps/cda/02-tools.html), we build the tool system that lets the agent actually *do* things.